# Dataset benchmark view

This notebook is for benchmark runs that include named real datasets. It excludes the synthetic `blobs` dataset completely and groups results primarily by `Dataset`.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PYTHON_SRC = Path.cwd() / "python"
if str(PYTHON_SRC) not in sys.path:
    sys.path.insert(0, str(PYTHON_SRC))

from benchmark_reporting.constants import *
from benchmark_reporting.io import (
    load_benchmark_summary,
    load_benchmark_data,
    load_cachegrind_summary,
    load_exclusion_summary,
    load_speedup_summary,
)

SUMMARY_JSON = Path("datasets/benchmark_summary.json")
EXCLUDED_DATASETS = {"blobs"}

def exclude_synthetic_datasets(df: pd.DataFrame, *, dataset_col: str = COL_DATASET) -> pd.DataFrame:
    """Return only non-synthetic dataset rows.

    Older summaries without a dataset column are treated as legacy synthetic-only
    summaries, so this notebook shows no rows for them.
    """
    if df.empty:
        return df
    if dataset_col not in df.columns:
        return df.iloc[0:0].copy()
    return df[~df[dataset_col].astype(str).isin(EXCLUDED_DATASETS)].copy()


In [ ]:
summary = load_benchmark_summary(SUMMARY_JSON)
configs = pd.DataFrame(summary.get("configs", []))
configs = exclude_synthetic_datasets(configs, dataset_col="dataset")

if configs.empty:
    display(Markdown("No non-synthetic dataset configs found in the benchmark summary."))
else:
    overview = (
        configs[["dataset", "dimensions", "samples", "clusters", "config_id"]]
        .rename(
            columns={
                "dataset": COL_DATASET,
                "dimensions": COL_DIMENSIONS,
                "samples": COL_SAMPLES,
                "clusters": COL_CLUSTERS,
                "config_id": "Config ID",
            }
        )
        .drop_duplicates()
        .sort_values([COL_DATASET, COL_DIMENSIONS, COL_SAMPLES, COL_CLUSTERS])
        .reset_index(drop=True)
    )
    display(Markdown("## Dataset/config overview"))
    display(overview)


In [ ]:
benchmarks = exclude_synthetic_datasets(load_benchmark_data(SUMMARY_JSON))

if benchmarks.empty:
    display(Markdown("No non-synthetic benchmark timing rows found."))
else:
    timing_cols = [
        COL_DATASET,
        COL_PHASE,
        COL_STAGE,
        COL_VARIANT,
        COL_PARAMS,
        COL_REFERENCE,
        COL_LANGUAGE,
        COL_DIMENSIONS,
        COL_SAMPLES,
        COL_CLUSTERS,
        COL_TIME_S,
    ]
    display(Markdown("## Timing rows by dataset"))
    display(
        benchmarks[timing_cols]
        .sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS, COL_LANGUAGE])
        .reset_index(drop=True)
    )


In [ ]:
speedups = exclude_synthetic_datasets(load_speedup_summary(SUMMARY_JSON))

required_cols = {
    COL_DATASET,
    COL_PHASE,
    COL_SPEEDUP,
    COL_SPEEDUP_CI_LOW,
    COL_SPEEDUP_CI_HIGH,
}

if speedups.empty:
    display(Markdown("No non-synthetic speedup rows found."))
elif required_cols.issubset(speedups.columns):
    label_cols = [
        col
        for col in (COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS)
        if col in speedups.columns
    ]

    plot_data = speedups.copy()

    group_cols = [COL_DATASET] + label_cols

    plot_data = (
        plot_data.groupby(group_cols, observed=True, dropna=False)
        .agg(
            speedup=(COL_SPEEDUP, "median"),
            ci_low=(COL_SPEEDUP_CI_LOW, "median"),
            ci_high=(COL_SPEEDUP_CI_HIGH, "median"),
        )
        .reset_index()
        .sort_values(group_cols)
    )

    def make_bar_label(row):
        parts = []
        for col in label_cols:
            value = row[col]
            if value is None or str(value) == "nan" or str(value) == "":
                continue
            if col == COL_PHASE:
                continue
            parts.append(str(value))

        suffix = " / ".join(parts)
        return f"{row[COL_PHASE]}" if not suffix else f"{row[COL_PHASE]}\n{suffix}"

    plot_data["bar_label"] = plot_data.apply(make_bar_label, axis=1)

    datasets = list(plot_data[COL_DATASET].drop_duplicates())
    bar_labels = list(plot_data["bar_label"].drop_duplicates())
    phases = list(plot_data[COL_PHASE].drop_duplicates())

    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    phase_colors = {
        phase: color_cycle[i % len(color_cycle)]
        for i, phase in enumerate(phases)
    }

    x = np.arange(len(datasets))
    width = 0.8 / max(len(bar_labels), 1)

    fig, ax = plt.subplots(figsize=(max(10, 1.5 * len(bar_labels)), 5))

    for i, bar_label in enumerate(bar_labels):
        subset = (
            plot_data[plot_data["bar_label"] == bar_label]
            .set_index(COL_DATASET)
            .reindex(datasets)
        )

        y = subset["speedup"].to_numpy(dtype=float)
        low = subset["ci_low"].to_numpy(dtype=float)
        high = subset["ci_high"].to_numpy(dtype=float)

        lower_err = np.maximum(y - low, 0.0)
        upper_err = np.maximum(high - y, 0.0)

        # Same color for all variants belonging to the same phase.
        phase_values = subset[COL_PHASE].dropna()
        if phase_values.empty:
            continue
        phase = phase_values.iloc[0]

        offset = (i - (len(bar_labels) - 1) / 2) * width

        ax.bar(
            x + offset,
            y,
            width,
            label=bar_label,
            color=phase_colors[phase],
            yerr=np.vstack([lower_err, upper_err]),
            capsize=4,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha="right")
    ax.set_ylabel(COL_SPEEDUP)
    ax.set_title("Speedup by dataset, phase, and variant")
    ax.axhline(1.0, linewidth=1)

    ax.legend(
        title="Phase / variant",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0,
    )

    fig.tight_layout()
else:
    missing = sorted(required_cols - set(speedups.columns))
    print(f"Cannot plot speedup CI bars. Missing columns: {missing}")


In [ ]:
cachegrind = exclude_synthetic_datasets(load_cachegrind_summary(SUMMARY_JSON))

if cachegrind.empty:
    display(Markdown("No non-synthetic Cachegrind rows found."))
else:
    cachegrind_cols = [
        COL_DATASET,
        COL_PHASE,
        COL_STAGE,
        COL_VARIANT,
        COL_PARAMS,
        COL_DIMENSIONS,
        COL_SAMPLES,
        COL_CLUSTERS,
        COL_CACHEGRIND_IR,
        COL_CACHEGRIND_D1_DATA_MISS_RATE,
        COL_CACHEGRIND_LL_DATA_MISS_RATE,
    ]
    available_cols = [col for col in cachegrind_cols if col in cachegrind.columns]
    display(Markdown("## Cachegrind by dataset"))
    display(
        cachegrind[available_cols]
        .sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_VARIANT, COL_PARAMS])
        .reset_index(drop=True)
    )


In [ ]:
exclusions = exclude_synthetic_datasets(load_exclusion_summary(SUMMARY_JSON))

if exclusions.empty:
    display(Markdown("No non-synthetic configured exclusions found."))
else:
    display(Markdown("## Exclusions by dataset"))
    display(
        exclusions.sort_values([COL_DATASET, COL_PHASE, COL_STAGE, COL_DIMENSIONS, COL_SAMPLES, COL_CLUSTERS])
        .reset_index(drop=True)
    )
